In [2]:
import pandas as pd
import time
from geopy.geocoders import Nominatim

# Load CSV files
df_konvensional = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Backup/ojk_cfs_bpr_konvensional.csv')
df_syariah = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Backup/ojk_cfs_bpr_syariah.csv')

print("Data Konvensional shape:", df_konvensional.shape)
print("Data Syariah shape:", df_syariah.shape)
print("\nKolom:", df_konvensional.columns.tolist())

Data Konvensional shape: (1863, 4)
Data Syariah shape: (194, 4)

Kolom: ['Provinsi', 'Kabupaten/Kota', 'Nama Bank', 'Kode Bank']


In [35]:
import pandas as pd
from geopy.geocoders import Nominatim
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# Load CSV files
print('Loading CSV files...')
df_konvensional = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Backup/ojk_cfs_bpr_konvensional.csv')
df_syariah = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Backup/ojk_cfs_bpr_syariah.csv')

print(f"Data Konvensional: {df_konvensional.shape}")
print(f"Data Syariah: {df_syariah.shape}")

# Extract unique banks
banks_konvensional = df_konvensional[['Nama Bank', 'Provinsi', 'Kabupaten/Kota']].drop_duplicates()
banks_syariah = df_syariah[['Nama Bank', 'Provinsi', 'Kabupaten/Kota']].drop_duplicates()

all_banks = pd.concat([
    banks_konvensional.assign(Jenis='Konvensional'),
    banks_syariah.assign(Jenis='Syariah')
], ignore_index=True)

all_banks = pd.read_csv('BankLongLat_FAILED.csv')

print(f"Total bank konvensional unik: {len(banks_konvensional)}")
print(f"Total bank syariah unik: {len(banks_syariah)}")
print(f"Total bank unik: {len(all_banks)}")

Loading CSV files...
Data Konvensional: (1863, 4)
Data Syariah: (194, 4)
Total bank konvensional unik: 1861
Total bank syariah unik: 194
Total bank unik: 3


In [36]:
all_banks

,Nama Bank,Provinsi,Kabupaten/Kota,Jenis,Latitude,Longitude
0,PD. BPR Beber,Provinsi Jawa Barat,Kab. Cirebon,Konvensional,NaN,NaN
1,PT Bank Perekonomian Rakyat Mulyo Lumintu,Provinsi Jawa Tengah,Kab. Magelang,Konvensional,NaN,NaN
2,PT Bank Perekonomian Rakyat Pembangunan Kerinci,Provinsi Jambi,Kota Sungai Penuh,Konvensional,NaN,NaN


In [37]:
# GEOCODING PAKE SELENIUM + GOOGLE MAPS (PARALLEL + HEADLESS)
# pip install selenium

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import re
import random
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Random User Agents
USER_AGENTS = [
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:121.0) Gecko/20100101 Firefox/121.0',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0',
]

def extract_coordinates_from_url(url):
    """Extract lat, lon dari Google Maps URL"""
    # Pattern: @-6.234900,106.989600
    pattern = r'@(-?\d+\.\d+),(-?\d+\.\d+)'
    match = re.search(pattern, url)
    if match:
        return float(match.group(1)), float(match.group(2))
    return None, None

def create_driver():
    """Create new Firefox driver instance"""
    firefox_options = Options()
    
    # HEADLESS MODE (ga muncul browser window)
    firefox_options.add_argument('--headless')
    
    # PRIVATE BROWSING (Incognito)
    firefox_options.add_argument('-private')
    
    # ANTI-DETECTION
    firefox_options.set_preference("dom.webdriver.enabled", False)
    firefox_options.set_preference('useAutomationExtension', False)
    firefox_options.set_preference("general.useragent.override", random.choice(USER_AGENTS))
    
    # PRIVACY & SECURITY
    firefox_options.set_preference("privacy.trackingprotection.enabled", True)
    firefox_options.set_preference("privacy.donottrackheader.enabled", True)
    firefox_options.set_preference("privacy.firstparty.isolate", True)
    firefox_options.set_preference("network.cookie.cookieBehavior", 1)
    firefox_options.set_preference("network.http.referer.spoofSource", True)
    
    # DISABLE NOTIFICATIONS & GEOLOCATION
    firefox_options.set_preference("dom.webnotifications.enabled", False)
    firefox_options.set_preference("geo.enabled", False)
    firefox_options.set_preference("geo.provider.use_corelocation", False)
    
    # FASTER LOADING (disable images)
    firefox_options.set_preference("permissions.default.image", 2)
    
    # DISABLE CACHE
    firefox_options.set_preference("browser.cache.disk.enable", False)
    firefox_options.set_preference("browser.cache.memory.enable", False)
    firefox_options.set_preference("browser.cache.offline.enable", False)
    
    # Setup geckodriver path
    service = Service(executable_path='/opt/homebrew/bin/geckodriver')
    
    return webdriver.Firefox(service=service, options=firefox_options)

def geocode_google_maps_selenium(bank_name, city, province):
    """Geocode dengan Selenium + Google Maps"""
    driver = None
    try:
        # Create driver
        driver = create_driver()
        wait = WebDriverWait(driver, 15)
        
        # Build query
        query = f"{bank_name}, {city}, {province}, Indonesia"
        
        # Go to Google Maps
        driver.get("https://www.google.com/maps")
        time.sleep(2)
        
        # Find search box dan search
        search_box = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="searchboxinput"]')))
        search_box.clear()
        search_box.send_keys(query)
        search_box.send_keys(Keys.RETURN)
        
        # Wait for results to load
        time.sleep(3)
        
        # Try to click first result (div dengan class yang contains result)
        try:
            # Cari first result link - biasanya ada di div dengan attribute aria-label
            first_result = wait.until(EC.element_to_be_clickable((
                By.CSS_SELECTOR, 
                'a[href*="@"]'  # Link yang ada koordinat di href
            )))
            first_result.click()
            time.sleep(2)
        except (TimeoutException, NoSuchElementException):
            # Kalo ga bisa click, langsung extract dari URL
            pass
        
        # Extract coordinates dari URL
        current_url = driver.current_url
        lat, lon = extract_coordinates_from_url(current_url)
        
        if lat and lon:
            # Validasi Indonesia
            if -15 <= lat <= 10 and 90 <= lon <= 150:
                return lat, lon, "SUCCESS"
        
    except Exception as e:
        return None, None, "FAILED"
    
    finally:
        if driver:
            driver.quit()
    
    return None, None, "FAILED"

print("\nGEOCODING DENGAN SELENIUM + GOOGLE MAPS (PARALLEL + HEADLESS)")

all_banks['Latitude'] = None
all_banks['Longitude'] = None

berhasil = 0
gagal_count = 0
processed = 0
results = {}

# Parallel processing dengan ThreadPoolExecutor
max_workers = 8 # 3 browsers parallel

try:
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {}
        for idx, row in all_banks.iterrows():
            future = executor.submit(
                geocode_google_maps_selenium,
                row['Nama Bank'],
                row['Kabupaten/Kota'],
                row['Provinsi']
            )
            futures[future] = idx
        
        # Process completed tasks
        for future in tqdm(as_completed(futures), total=len(futures)):
            idx = futures[future]
            lat, lon, status = future.result()
            
            results[idx] = (lat, lon, status)
            processed += 1
            
            row = all_banks.iloc[idx]
            
            if status == "SUCCESS":
                berhasil += 1
                if berhasil <= 30:
                    print(f"✓ [{berhasil}] {row['Nama Bank'][:40]} | ({lat:.6f}, {lon:.6f})")
            else:
                gagal_count += 1
                if gagal_count <= 20:
                    print(f"✗ [{gagal_count}] {row['Nama Bank'][:40]}")
            
            # Auto-save tiap 50 bank
            if processed % 50 == 0:
                for i, (lat, lon, status) in results.items():
                    all_banks.at[i, 'Latitude'] = lat
                    all_banks.at[i, 'Longitude'] = lon
                all_banks.to_csv('BankLongLat_PROGRESS.csv', index=False)
                print(f"\n SAVED: {processed}/{len(all_banks)} | Success: {berhasil} | Fail: {gagal_count}\n")

except KeyboardInterrupt:
    print("\n STOPPED!")

# Final save
for idx, (lat, lon, status) in sorted(results.items()):
    all_banks.at[idx, 'Latitude'] = lat
    all_banks.at[idx, 'Longitude'] = lon

all_banks.to_csv('BankLongLat_PROGRESS.csv', index=False)

print(f"\n{'='*70}")
print(f"HASIL GEOCODING SELENIUM + GOOGLE MAPS:")
print(f"{'='*70}")
print(f"Total: {len(all_banks)}")
print(f"Processed: {processed}/{len(all_banks)} ({processed/len(all_banks)*100:.1f}%)")
print(f"✓ Success: {berhasil} ({berhasil/len(all_banks)*100:.1f}%)")
print(f"✗ Failed: {gagal_count} ({gagal_count/len(all_banks)*100:.1f}%)")
print(f"Success Rate: {berhasil/processed*100:.1f}%")
print(f"{'='*70}")
print(f"\n Saved to: BankLongLat_PROGRESS.csv")


GEOCODING DENGAN SELENIUM + GOOGLE MAPS (PARALLEL + HEADLESS)


 33%|███▎      | 1/3 [00:27<00:54, 27.03s/it]

✓ [1] PT Bank Perekonomian Rakyat Mulyo Lumint | (-7.581955, 110.282883)


100%|██████████| 3/3 [00:28<00:00,  9.48s/it]

✓ [2] PT Bank Perekonomian Rakyat Pembangunan  | (-2.065917, 101.394862)
✓ [3] PD. BPR Beber | (-6.795517, 108.477517)

HASIL GEOCODING SELENIUM + GOOGLE MAPS:
Total: 3
Processed: 3/3 (100.0%)
✓ Success: 3 (100.0%)
✗ Failed: 0 (0.0%)
Success Rate: 100.0%

 Saved to: BankLongLat_PROGRESS.csv


In [38]:
all_banks[(all_banks['Latitude'].isnull()) & (all_banks['Longitude'].isnull())].to_csv('BankLongLat_FAILED.csv', index=False)

In [39]:
all_banks

,Nama Bank,Provinsi,Kabupaten/Kota,Jenis,Latitude,Longitude
0,PD. BPR Beber,Provinsi Jawa Barat,Kab. Cirebon,Konvensional,-6.795517,108.477517
1,PT Bank Perekonomian Rakyat Mulyo Lumintu,Provinsi Jawa Tengah,Kab. Magelang,Konvensional,-7.581955,110.282883
2,PT Bank Perekonomian Rakyat Pembangunan Kerinci,Provinsi Jambi,Kota Sungai Penuh,Konvensional,-2.065917,101.394862


In [7]:
all_banks.to_csv('BankLongLat.csv', index=False)

In [40]:
all_banks

,Nama Bank,Provinsi,Kabupaten/Kota,Jenis,Latitude,Longitude
0,PD. BPR Beber,Provinsi Jawa Barat,Kab. Cirebon,Konvensional,-6.795517,108.477517
1,PT Bank Perekonomian Rakyat Mulyo Lumintu,Provinsi Jawa Tengah,Kab. Magelang,Konvensional,-7.581955,110.282883
2,PT Bank Perekonomian Rakyat Pembangunan Kerinci,Provinsi Jambi,Kota Sungai Penuh,Konvensional,-2.065917,101.394862


In [41]:
df = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Main/BankLongLat.csv')

# Create a merge key for matching
all_banks['merge_key'] = all_banks['Nama Bank'] + '|' + all_banks['Provinsi'] + '|' + all_banks['Kabupaten/Kota']
df['merge_key'] = df['Nama Bank'] + '|' + df['Provinsi'] + '|' + df['Kabupaten/Kota']

# Update df with coordinates from all_banks
df_updated = df.drop(columns=['Latitude', 'Longitude']).merge(
    all_banks[['merge_key', 'Latitude', 'Longitude']], 
    on='merge_key', 
    how='left'
)

# If there are still missing coordinates in df_updated, keep original df coordinates
df_updated['Latitude'] = df_updated['Latitude'].fillna(df['Latitude'])
df_updated['Longitude'] = df_updated['Longitude'].fillna(df['Longitude'])

# Drop merge key and reorder columns
df_updated = df_updated.drop(columns=['merge_key'])
df_updated = df_updated[['Nama Bank', 'Provinsi', 'Kabupaten/Kota', 'Jenis', 'Latitude', 'Longitude']]

# Save updated dataframe
df_updated.to_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Main/BankLongLat.csv', index=False)

print(f"Original df - Missing coordinates: {df['Latitude'].isnull().sum()}")
print(f"Updated df - Missing coordinates: {df_updated['Latitude'].isnull().sum()}")
print(f"Coordinates updated: {df['Latitude'].isnull().sum() - df_updated['Latitude'].isnull().sum()}")

Original df - Missing coordinates: 3
Updated df - Missing coordinates: 0
Coordinates updated: 3


/var/folders/m3/3gmfjhln13qbt6p_47473gs40000gn/T/ipykernel_10174/748032985.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_updated['Latitude'] = df_updated['Latitude'].fillna(df['Latitude'])
/var/folders/m3/3gmfjhln13qbt6p_47473gs40000gn/T/ipykernel_10174/748032985.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_updated['Longitude'] = df_updated['Longitude'].fillna(df['Longitude'])


In [33]:
df = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Main/BankLongLat.csv')

In [34]:
df.isnull().sum()

Nama Bank         0
Provinsi          0
Kabupaten/Kota    0
Jenis             0
Latitude          3
Longitude         3
dtype: int64